In [1]:
import logging

import teehr
from teehr.evaluation.evaluation import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.fetching.utils import format_nwm_configuration_metadata
from teehr import Variable

logger = logging.getLogger()

In [2]:
spark = create_spark_session(
    aws_profile="admin-user"
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS credentials from ~/.aws/credentials profile 'admin-user
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!
INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


In [3]:
ev.configurations.to_sdf().select("name", "description", "timeseries_type").show(truncate=False, n=100)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.


+------------------------------------------+--------------------------------------------------------------------------------------------------+---------------+
|name                                      |description                                                                                       |timeseries_type|
+------------------------------------------+--------------------------------------------------------------------------------------------------+---------------+
|nwm30_forcing_analysis_assim_extend_alaska|Alaska StageIV mean areal forcing for NWM extended analysis                                       |primary        |
|nwm30_forcing_analysis_assim_extend       |CONUS STAGEIV mean areal forcing for NWM extended analysis                                        |primary        |
|nwm30_forcing_analysis_assim_alaska       |Alaska MRMS mean areal forcing for NWM standard analysis                                          |primary        |
|nwm30_forcing_analysis_assim           

In [13]:
nwm_version = "nwm31"

secondary_nwm_config_names_to_add = [
    "short_range",
    "medium_range_mem1",
    "analysis_assim_no_da",
    "analysis_assim_alaska_no_da",
    "analysis_assim_extend_no_da",
    "analysis_assim_extend_alaska_no_da",
    "analysis_assim_hawaii_no_da",
    "analysis_assim_puertorico_no_da",
    "short_range_alaska",
    "short_range_hawaii",
    "short_range_puertorico",
    "medium_range_alaska_mem1",
    "medium_range_blend",
    "medium_range_blend_alaska",
    "forcing_short_range",
    "forcing_short_range_alaska",
    "forcing_short_range_hawaii",
    "forcing_short_range_puertorico",
    "forcing_medium_range",
    "forcing_medium_range_alaska",
    "forcing_medium_range_blend",
    "forcing_medium_range_blend_alaska"
    "forcing_medium_range_hawaii",
    "forcing_medium_range_puertorico"    
]

primary_nwm_config_names_to_add = [
    "forcing_analysis_assim",
    "forcing_analysis_assim_alaska",
    "forcing_analysis_assim_extend",
    "forcing_analysis_assim_extend_alaska",
    "forcing_analysis_assim_hawaii",
    "forcing_analysis_assim_puertorico",
]

In [14]:
%%time
for nwm_configuration in secondary_nwm_config_names_to_add:
    ev_config = format_nwm_configuration_metadata(
        nwm_config_name=nwm_configuration,
        nwm_version=nwm_version
    )
    print(ev_config)
    ev.configurations.add(
        configuration=[
            teehr.Configuration(
                name=ev_config["name"],
                timeseries_type="secondary",
                description=ev_config["description"],
            )
        ]
    )

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.


{'name': 'nwm31_medium_range', 'member': '1', 'description': 'CONUS NWM medium range, GFS forcing'}


INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 5.470 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'configurations'.
INFO:teehr.evaluation.tables.generic_table:Getting table: configurations.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.write:Finished writing to warehouse table 'configurations' in 4.729 seconds.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.


{'name': 'nwm31_medium_range_alaska', 'member': '1', 'description': 'Alaska NWM medium range, GFS forcing'}


INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 5.415 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'configurations'.
INFO:teehr.evaluation.tables.generic_table:Getting table: configurations.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.write:Finished writing to warehouse table 'configurations' in 4.408 seconds.


CPU times: user 85.8 ms, sys: 24.3 ms, total: 110 ms
Wall time: 20.2 s


In [15]:
%%time
for nwm_configuration in primary_nwm_config_names_to_add:
    ev_config = format_nwm_configuration_metadata(
        nwm_config_name=nwm_configuration,
        nwm_version=nwm_version
    )
    ev.configurations.add(
        configuration=[
            teehr.Configuration(
                name=ev_config["name"],
                timeseries_type="primary",
                description=ev_config["description"],
            )
        ]
    )

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.validate:Start enforcing dataframe schema.
INFO:teehr.evaluation.validate:Validating DataFrame against schema.
INFO:teehr.evaluation.validate:Finished enforcing dataframe schema in 5.386 seconds.
INFO:teehr.evaluation.write:Start writing to warehouse table 'configurations'.
INFO:teehr.evaluation.tables.generic_table:Getting table: configurations.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.write:Finished writing to warehouse table 'configurations' in 4.016 seconds.
INFO:teehr.evaluation.tables

CPU times: user 58.3 ms, sys: 50.5 ms, total: 109 ms
Wall time: 19.3 s


In [25]:
ev.configurations.to_sdf().select("name", "description", "timeseries_type").orderBy("name").show(n=200, truncate=False)

INFO:teehr.evaluation.tables.base_table:Initializing Table for table: configurations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.configurations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.configurations.


+------------------------------------------+--------------------------------------------------------------------------------------------------+---------------+
|name                                      |description                                                                                       |timeseries_type|
+------------------------------------------+--------------------------------------------------------------------------------------------------+---------------+
|nrds_v22_cfenom_medium_range              |NRDS DataStream medium range forecasts, hydrofabric v.2.2, CFE-NOM                                |secondary      |
|nrds_v22_cfenom_short_range               |POC version of DataStream forecasts, hydrofabric v.2.2, CFE-NOM                                   |secondary      |
|nrds_v22_lstm0_medium_range               |NRDS DataStream medium range forecasts, hydrofabric v.2.2, LSTM_0                                 |secondary      |
|nrds_v22_lstm0_short_range             